# 🌐 Global Financial Markets Dataset (2000 - Present)
**Stocks | Crypto | Commodities | Forex**

Auto-updates daily with the latest market data.  
Data always ends on **yesterday's date** to guarantee no `NaN` closing prices.

---
**Assets covered:** 34 assets across 4 classes  
**Source:** Yahoo Finance via `yfinance`  
**Update cadence:** Run notebook daily on Kaggle


## INSTALL LIBRARIES

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'pandas', 'numpy', 'yfinance', 'requests', 'urllib3'])
print('✅ All libraries installed!')

✅ All libraries installed!


## IMPORTS & DATE SETUP

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from datetime import datetime, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

START_DATE = '2000-01-01'
END_DATE   = datetime.utcnow().strftime('%Y-%m-%d')

print('✅ Imports successful!')
print(f'📅 Date window : {START_DATE}  →  {END_DATE}')
print(f'⏰ Script start: {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")}')

✅ Imports successful!
📅 Date window : 2000-01-01  →  2026-10-21
⏰ Script start: 2026-10-21 18:49:06 UTC


## SHARED HELPER

In [3]:
def _flatten_yf(df):
    """
    Recent yfinance versions return MultiIndex columns, e.g.:
        [('Open', 'AAPL'), ('Close', 'AAPL'), ...]
    After reset_index() the Date column also gets a tuple header.
    We drop level-1 (ticker name) — it's redundant because we
    add a 'Symbol' column ourselves.
    """
    df = df.reset_index()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    return df

## STOCK INDICES

In [4]:
def download_stocks():
    print('\n' + '='*70)
    print('[1/4] DOWNLOADING STOCK INDICES DATA')
    print('='*70)

    stocks = {
        'S&P500':    '^GSPC',
        'NASDAQ':    '^IXIC',
        'NIFTY50':   '^NSEI',
        'DAX':       '^GDAXI',
        'FTSE100':   '^FTSE',
        'Nikkei225': '^N225',
        'HSI':       '^HSI',
        'Shanghai':  '000001.SS',
        'SENSEX':    '^BSESN',
        'KOSPI':     '^KS11',
    }

    rows, count = [], 0
    for name, symbol in stocks.items():
        try:
            print(f'  → {name:<15} ', end='', flush=True)
            raw = yf.download(symbol, start=START_DATE, end=END_DATE, progress=False)
            if raw.empty:
                print('⚠️  No data'); continue
            df = _flatten_yf(raw)
            df['Symbol']     = symbol
            df['Asset_Name'] = name
            df['Asset_Type'] = 'Stock Index'
            df['Region']     = 'Global'
            rows.append(df); count += len(df)
            print(f'✅ ({len(df):,} records)')
        except Exception as e:
            print(f'⚠️  Skipped ({str(e)[:50]})')

    result = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    print(f'\n✅ Total Stock Records: {count:,}')
    return result

stocks_df = download_stocks()


[1/4] DOWNLOADING STOCK INDICES DATA
  → S&P500          ✅ (6,740 records)
  → NASDAQ          ✅ (6,740 records)
  → NIFTY50         ✅ (4,683 records)
  → DAX             ✅ (6,806 records)
  → FTSE100         ✅ (6,770 records)
  → Nikkei225       ✅ (6,562 records)
  → HSI             ✅ (6,602 records)
  → Shanghai        ✅ (6,485 records)
  → SENSEX          ✅ (6,606 records)
  → KOSPI           ✅ (6,597 records)

✅ Total Stock Records: 64,591


## CRYPTOCURRENCY 

In [5]:
def download_crypto():
    print('\n' + '='*70)
    print('[2/4] DOWNLOADING CRYPTOCURRENCY DATA  (via yfinance)')
    print('='*70)

    cryptos = {
        'Bitcoin':  'BTC-USD',
        'Ethereum': 'ETH-USD',
        'Cardano':  'ADA-USD',
        'Solana':   'SOL-USD',
        'Litecoin': 'LTC-USD',
        'Dogecoin': 'DOGE-USD',
        'Ripple':   'XRP-USD',
        'BNB':      'BNB-USD',
    }

    rows, count = [], 0
    for name, symbol in cryptos.items():
        try:
            print(f'  → {name:<15} ', end='', flush=True)
            raw = yf.download(symbol, start=START_DATE, end=END_DATE, progress=False)
            if raw.empty:
                print('⚠️  No data'); continue
            df = _flatten_yf(raw)
            df['Symbol']     = symbol
            df['Asset_Name'] = name
            df['Asset_Type'] = 'Cryptocurrency'
            df['Region']     = 'Global'
            rows.append(df); count += len(df)
            print(f'✅ ({len(df):,} records)')
        except Exception as e:
            print(f'⚠️  Skipped ({str(e)[:50]})')

    result = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    print(f'\n✅ Total Crypto Records: {count:,}')
    return result

crypto_df = download_crypto()


[2/4] DOWNLOADING CRYPTOCURRENCY DATA  (via yfinance)
  → Bitcoin         ✅ (4,417 records)
  → Ethereum        ✅ (3,268 records)
  → Cardano         ✅ (3,268 records)
  → Solana          ✅ (2,385 records)
  → Litecoin        ✅ (4,417 records)
  → Dogecoin        ✅ (3,268 records)
  → Ripple          ✅ (3,268 records)
  → BNB             ✅ (3,268 records)

✅ Total Crypto Records: 27,559


## COMMODITIES

In [6]:
def download_commodities():
    print('\n' + '='*70)
    print('[3/4] DOWNLOADING COMMODITIES DATA')
    print('='*70)

    commodities = {
        'Crude Oil (WTI)': 'CL=F',
        'Brent Crude':     'BZ=F',
        'Gold':            'GC=F',
        'Natural Gas':     'NG=F',
        'Copper':          'HG=F',
        'Silver':          'SI=F',
        'Wheat':           'ZW=F',
        'Corn':            'ZC=F',
    }

    rows, count = [], 0
    for name, symbol in commodities.items():
        try:
            print(f'  → {name:<22} ', end='', flush=True)
            raw = yf.download(symbol, start=START_DATE, end=END_DATE, progress=False)
            if raw.empty:
                print('⚠️  No data'); continue
            df = _flatten_yf(raw)
            df['Symbol']     = symbol
            df['Asset_Name'] = name
            df['Asset_Type'] = 'Commodity'
            df['Region']     = 'Global'
            rows.append(df); count += len(df)
            print(f'✅ ({len(df):,} records)')
        except Exception as e:
            print(f'⚠️  Skipped ({str(e)[:50]})')

    result = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    print(f'\n✅ Total Commodity Records: {count:,}')
    return result

commodities_df = download_commodities()


[3/4] DOWNLOADING COMMODITIES DATA
  → Crude Oil (WTI)        ✅ (6,567 records)
  → Brent Crude            ✅ (4,786 records)
  → Gold                   ✅ (6,560 records)
  → Natural Gas            ✅ (6,565 records)
  → Copper                 ✅ (6,564 records)
  → Silver                 ✅ (6,561 records)
  → Wheat                  ✅ (6,583 records)
  → Corn                   ✅ (6,571 records)

✅ Total Commodity Records: 50,757


## FOREX / CURRENCIES

In [7]:
def download_forex():
    print('\n' + '='*70)
    print('[4/4] DOWNLOADING CURRENCY / FOREX DATA')
    print('='*70)

    currencies = {
        'EUR/USD': 'EURUSD=X',
        'GBP/USD': 'GBPUSD=X',
        'INR/USD': 'INRUSD=X',
        'JPY/USD': 'JPYUSD=X',
        'CHF/USD': 'CHFUSD=X',
        'AUD/USD': 'AUDUSD=X',
        'CAD/USD': 'CADUSD=X',
        'CNY/USD': 'CNYUSD=X',
    }

    rows, count = [], 0
    for name, symbol in currencies.items():
        try:
            print(f'  → {name:<15} ', end='', flush=True)
            raw = yf.download(symbol, start=START_DATE, end=END_DATE, progress=False)
            if raw.empty:
                print('⚠️  No data'); continue
            df = _flatten_yf(raw)
            df['Symbol']     = symbol
            df['Asset_Name'] = name
            df['Asset_Type'] = 'Currency'
            df['Region']     = 'Global'
            rows.append(df); count += len(df)
            print(f'✅ ({len(df):,} records)')
        except Exception as e:
            print(f'⚠️  Skipped ({str(e)[:50]})')

    result = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    print(f'\n✅ Total Forex Records: {count:,}')
    return result

forex_df = download_forex()


[4/4] DOWNLOADING CURRENCY / FOREX DATA
  → EUR/USD         ✅ (5,938 records)
  → GBP/USD         ✅ (5,950 records)
  → INR/USD         ✅ (5,935 records)
  → JPY/USD         ✅ (6,958 records)
  → CHF/USD         ✅ (6,004 records)
  → AUD/USD         ✅ (5,314 records)
  → CAD/USD         ✅ (6,006 records)
  → CNY/USD         ✅ (6,340 records)

✅ Total Forex Records: 48,445


## COMBINE & CLEAN

In [8]:
print('\n' + '='*70)
print('COMBINING & CLEANING DATA')
print('='*70)

all_dfs = [df for df in [stocks_df, crypto_df, commodities_df, forex_df]
           if not df.empty]

if not all_dfs:
    raise RuntimeError('⚠️  No data collected — check your internet connection!')

all_data = pd.concat(all_dfs, ignore_index=True)

# Safety-net: flatten any residual MultiIndex after concat
if isinstance(all_data.columns, pd.MultiIndex):
    all_data.columns = [col[0] for col in all_data.columns]

# Normalise to lowercase
all_data.columns = [str(c).lower() for c in all_data.columns]

# Keep only required columns
required = ['date', 'open', 'high', 'low', 'close', 'volume',
            'symbol', 'asset_name', 'asset_type', 'region']
all_data = all_data[[c for c in required if c in all_data.columns]]

# Type coercions
all_data['date'] = pd.to_datetime(all_data['date'], errors='coerce')
all_data = all_data.dropna(subset=['date', 'close'])   # ← drop rows missing close
all_data = all_data.sort_values('date').reset_index(drop=True)
all_data = all_data.drop_duplicates(subset=['date', 'symbol'], keep='last')
all_data['region'] = all_data['region'].fillna('Global')

for col in ['open', 'high', 'low', 'close']:
    if col in all_data.columns:
        all_data[col] = pd.to_numeric(all_data[col], errors='coerce').round(4)

if 'volume' in all_data.columns:
    all_data['volume'] = pd.to_numeric(all_data['volume'], errors='coerce').fillna(0).astype('int64')

print(f'✅ Columns : {list(all_data.columns)}')
print('✅ Combined and cleaned!')


COMBINING & CLEANING DATA
✅ Columns : ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'asset_name', 'asset_type', 'region']
✅ Combined and cleaned!


## VALIDATION & STATISTICS

In [9]:
print('\n' + '='*70)
print('📊 DATASET STATISTICS')
print('='*70)
print(f'📈 Total Records  : {len(all_data):,}')
print(f'📅 Date Range     : {all_data["date"].min().date()}  →  {all_data["date"].max().date()}')
print(f'🏢 Asset Types    : {all_data["asset_type"].nunique()}')
print(f'💎 Total Assets   : {all_data["symbol"].nunique()}')
print(f'🔴 NaN in close   : {all_data["close"].isna().sum()}  (should be 0)')
print(f'⏰ Last Updated   : {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")}')

print('\n📋 Breakdown by Asset Type:')
for atype, grp in all_data.groupby('asset_type'):
    print(f'  • {atype:<20}: {len(grp):>8,} rows | {grp["symbol"].nunique()} assets')

print('\n🔝 Latest 5 Records:')
cols = [c for c in ['date', 'symbol', 'asset_name', 'asset_type', 'close', 'volume']
        if c in all_data.columns]
print(all_data.tail(5)[cols].to_string(index=False))


📊 DATASET STATISTICS
📈 Total Records  : 191,352
📅 Date Range     : 2000-01-03  →  2026-10-20
🏢 Asset Types    : 4
💎 Total Assets   : 34
🔴 NaN in close   : 0  (should be 0)
⏰ Last Updated   : 2026-10-21 18:49:21 UTC

📋 Breakdown by Asset Type:
  • Commodity           :   50,757 rows | 8 assets
  • Cryptocurrency      :   27,559 rows | 8 assets
  • Currency            :   48,445 rows | 8 assets
  • Stock Index         :   64,591 rows | 10 assets

🔝 Latest 5 Records:
      date    symbol asset_name     asset_type     close     volume
2026-10-20   BNB-USD        BNB Cryptocurrency  740.8236 1284997038
2026-10-20   LTC-USD   Litecoin Cryptocurrency   61.5316  243519193
2026-10-20      HG=F     Copper      Commodity    6.5665       1057
2026-10-20 000001.SS   Shanghai    Stock Index 3735.3701     408700
2026-10-20  CNYUSD=X    CNY/USD       Currency    0.1495          0


## SAVE TO CSV

In [10]:
output_path = '/kaggle/working/global_financial_markets_2000_Now.csv'
all_data.to_csv(output_path, index=False)

size_mb = all_data.memory_usage(deep=True).sum() / (1024 * 1024)
print(f'✅ Saved  →  {output_path}')
print(f'   Rows  : {len(all_data):,}')
print(f'   Cols  : {len(all_data.columns)}')
print(f'   Size  : ~{size_mb:.1f} MB (in-memory)')

print('\n' + '='*70)
print('🎉 DATASET UPDATE COMPLETE!')
print('='*70)

print('\n📋 Data Preview (first 10 rows):')
print(all_data.head(10).to_string(index=False))

✅ Saved  →  /kaggle/working/global_financial_markets_2000_Now.csv
   Rows  : 191,352
   Cols  : 10
   Size  : ~49.8 MB (in-memory)

🎉 DATASET UPDATE COMPLETE!

📋 Data Preview (first 10 rows):
      date       open       high        low      close     volume   symbol asset_name  asset_type region
2000-01-03  1469.2500  1478.0000  1438.3600  1455.2200  931800000    ^GSPC     S&P500 Stock Index Global
2000-01-03  6961.7202  7159.3301  6720.8701  6750.7598   43072500   ^GDAXI        DAX Stock Index Global
2000-01-03     0.0098     0.0099     0.0097     0.0098          0 JPYUSD=X    JPY/USD    Currency Global
2000-01-03 17057.6992 17426.1602 17057.6992 17369.6309          0     ^HSI        HSI Stock Index Global
2000-01-03  5209.5400  5384.6602  5209.5400  5375.1099          0   ^BSESN     SENSEX Stock Index Global
2000-01-03  4186.1899  4192.1899  3989.7100  4131.1499 1510070000    ^IXIC     NASDAQ Stock Index Global
2000-01-04 18937.4492 19187.6094 18937.4492 19002.8594          0    ^N22

---

## 👍 Found this dataset useful?

If this notebook or dataset helped your analysis, please consider giving it an **upvote** - it helps others discover it and motivates me to keep it updated daily! 🙏

**How to upvote:**
- 📌 Click the **▲ Upvote** button at the top of this notebook page
- ⭐ Also consider bookmarking the dataset so you get notified on updates

Feel free to leave a comment if you have suggestions or spot any issues - I read every one! 💬

 
<div style="text-align:center; margin-top:16px;">
  <a href="https://www.kaggle.com/datasets/belbino/global-financial-markets-dataset-2010-present" style="background:#6366f1; color:white; padding:10px 28px;
     border-radius:8px; text-decoration:none; font-weight:600; font-size:14px;">
    👍 Upvote on Kaggle
  </a>
</div>
